In [ ]:
import os
import json
import zipfile
import urllib.request
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

plt.rcParams["figure.dpi"] = 120
print("Environment initialized successfully.")

Environment initialized successfully.


In [ ]:
work_dir = "./coco_week5_audit"
img_dir = os.path.join(work_dir, "sample_images")
os.makedirs(img_dir, exist_ok=True)

ann_zip_path = os.path.join(work_dir, "annotations_trainval2017.zip")
ann_json_path = os.path.join(work_dir, "annotations", "instances_val2017.json")

if not os.path.exists(ann_json_path):
    url = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
    print("Downloading COCO 2017 annotations (~241 MB)...")
    urllib.request.urlretrieve(url, ann_zip_path)
    print("Extracting instances_val2017.json...")
    with zipfile.ZipFile(ann_zip_path, "r") as zf:
        zf.extract("annotations/instances_val2017.json", path=work_dir)
    os.remove(ann_zip_path)

print(f"Annotation JSON ready at: {ann_json_path}")

Extracting instances_val2017.json...
Annotation JSON ready at: ./coco_week5_audit/annotations/instances_val2017.json


In [ ]:
with open(ann_json_path, "r", encoding="utf-8") as f:
    coco_data = json.load(f)

print("COCO JSON loaded into memory.")
print("Top-level keys:", list(coco_data.keys()))

COCO JSON loaded into memory.
Top-level keys: ['info', 'licenses', 'images', 'annotations', 'categories']


In [ ]:
images = coco_data.get("images", [])
annotations = coco_data.get("annotations", [])
categories = coco_data.get("categories", [])

print("=== COCO Dataset Overview (val2017) ===")
print(f"Info description : {coco_data.get('info', {}).get('description', 'N/A')}")
print(f"Total Images     : {len(images):,}")
print(f"Total Annotations: {len(annotations):,}")
print(f"Total Categories : {len(categories):,}")
print(f"Avg Objects/Image: {len(annotations) / len(images):.2f}")

=== COCO Dataset Overview (val2017) ===
Info description : COCO 2017 Dataset
Total Images     : 5,000
Total Annotations: 36,781
Total Categories : 80
Avg Objects/Image: 7.36


In [ ]:
print("Sample 'images' record keys      :", list(images[0].keys()))
print("Sample 'annotations' record keys :", list(annotations[0].keys()))
print("Sample 'categories' record keys  :", list(categories[0].keys()))

Sample 'images' record keys      : ['license', 'file_name', 'coco_url', 'height', 'width', 'date_captured', 'flickr_url', 'id']
Sample 'annotations' record keys : ['segmentation', 'area', 'iscrowd', 'image_id', 'bbox', 'category_id', 'id']
Sample 'categories' record keys  : ['supercategory', 'id', 'name']


In [ ]:
cat_df = pd.DataFrame(categories).sort_values("id").reset_index(drop=True)
cat_id_to_name = dict(zip(cat_df["id"], cat_df["name"]))

print(f"Extracted {len(cat_df)} categories. Note non-contiguous IDs (min={cat_df['id'].min()}, max={cat_df['id'].max()}).")
display(cat_df.head(15))

# Filter standard traffic-relevant classes present in COCO
traffic_related = cat_df[cat_df["name"].isin([
    "person", "bicycle", "car", "motorcycle", "bus", "truck", "traffic light", "stop sign"
])]
print("\nTraffic-adjacent classes available in standard COCO:")
display(traffic_related)

Extracted 80 categories. Note non-contiguous IDs (min=1, max=90).


,supercategory,id,name
0,person,1,person
1,vehicle,2,bicycle
2,vehicle,3,car
3,vehicle,4,motorcycle
4,vehicle,5,airplane
5,vehicle,6,bus
6,vehicle,7,train
7,vehicle,8,truck
8,vehicle,9,boat
9,outdoor,10,traffic light



Traffic-adjacent classes available in standard COCO:


,supercategory,id,name
0,person,1,person
1,vehicle,2,bicycle
2,vehicle,3,car
3,vehicle,4,motorcycle
5,vehicle,6,bus
7,vehicle,8,truck
9,outdoor,10,traffic light
11,outdoor,13,stop sign


In [ ]:
img_df = pd.DataFrame(images)
img_id_to_info = {img["id"]: img for img in images}

print("=== Image Metadata Summary ===")
display(img_df[["id", "file_name", "width", "height"]].head(10))
print("\nResolution statistics:")
display(img_df[["width", "height"]].describe().round(1))

=== Image Metadata Summary ===


,id,file_name,width,height
0,397133,000000397133.jpg,640,427
1,37777,000000037777.jpg,352,230
2,252219,000000252219.jpg,640,428
3,87038,000000087038.jpg,640,480
4,174482,000000174482.jpg,640,388
5,403385,000000403385.jpg,640,511
6,6818,000000006818.jpg,427,640
7,480985,000000480985.jpg,375,500
8,458054,000000458054.jpg,640,426
9,331352,000000331352.jpg,351,500



Resolution statistics:


,width,height
count,5000.0,5000.0
mean,573.8,483.5
std,94.0,98.8
min,200.0,145.0
25%,500.0,426.0
50%,640.0,480.0
75%,640.0,517.8
max,640.0,640.0


In [ ]:
ann_df = pd.DataFrame(annotations)
ann_df["category_name"] = ann_df["category_id"].map(cat_id_to_name)

print("=== Sample Object Annotations ===")
display(ann_df[["id", "image_id", "category_id", "category_name", "bbox", "area", "iscrowd"]].head(10))

print("\nTop 10 Most Frequent Object Categories in COCO val2017:")
print(ann_df["category_name"].value_counts().head(10))

=== Sample Object Annotations ===


,id,image_id,category_id,category_name,bbox,area,iscrowd
0,1768,289343,18,dog,"[473.07, 395.93, 38.65, 28.67]",702.10575,0
1,1773,61471,18,dog,"[272.1, 200.23, 151.97, 279.77]",27718.47630,0
2,2551,472375,18,dog,"[124.71, 196.18, 372.85, 356.81]",78969.31690,0
3,3186,520301,18,dog,"[112.71, 154.82, 367.29, 479.35]",108316.66515,0
4,3419,579321,18,dog,"[200.61, 89.65, 400.22, 251.02]",75864.53530,0
5,3658,494869,18,dog,"[0.0, 421.09, 154.53, 208.61]",16932.13510,0
6,4594,554002,18,dog,"[427.58, 77.87, 188.88, 285.91]",32002.70300,0
7,7544,78823,18,dog,"[197.97, 117.22, 170.45, 222.07]",16694.40470,0
8,7915,419974,18,dog,"[61.68, 389.34, 130.77, 138.47]",9496.32990,0
9,7981,404484,18,dog,"[86.93, 90.76, 82.5, 74.54]",2613.77270,0



Top 10 Most Frequent Object Categories in COCO val2017:
category_name
person           11004
car               1932
chair             1791
book              1161
bottle            1025
cup                899
dining table       697
traffic light      637
bowl               626
handbag            540
Name: count, dtype: int64


In [ ]:
bbox_errors = []

for ann in annotations:
    img_info = img_id_to_info.get(ann["image_id"])
    if img_info is None:
        bbox_errors.append((ann["id"], "Orphaned image_id"))
        continue
    if ann["category_id"] not in cat_id_to_name:
        bbox_errors.append((ann["id"], "Invalid category_id"))
        continue

    x, y, w, h = ann["bbox"]
    img_w, img_h = img_info["width"], img_info["height"]

    # Check non-positive dimensions or out-of-bounds coordinates (with 1px float tolerance)
    if w <= 0 or h <= 0:
        bbox_errors.append((ann["id"], f"Non-positive box size: w={w}, h={h}"))
    elif x < -1 or y < -1 or (x + w) > (img_w + 1) or (y + h) > (img_h + 1):
        bbox_errors.append((ann["id"], f"Out of bounds: bbox={ann['bbox']} vs img=({img_w}, {img_h})"))

print(f"Total annotations audited: {len(annotations):,}")
print(f"Total integrity/geometry anomalies found: {len(bbox_errors)}")
if bbox_errors:
    print("Sample anomalies:", bbox_errors[:5])
else:
    print("PASS: All bounding boxes conform to [x_min, y_min, width, height] within image boundaries.")

Total annotations audited: 36,781
Total integrity/geometry anomalies found: 0
PASS: All bounding boxes conform to [x_min, y_min, width, height] within image boundaries.


In [ ]:
# Select 4 images that contain multiple traffic-adjacent objects (e.g., cars, people, traffic lights)
traffic_cat_ids = set(traffic_related["id"])
img_traffic_counts = Counter(
    ann["image_id"] for ann in annotations if ann["category_id"] in traffic_cat_ids
)

sample_img_ids = [img_id for img_id, _ in img_traffic_counts.most_common(4)]
sample_images = [img_id_to_info[img_id] for img_id in sample_img_ids]

for img_info in sample_images:
    local_path = os.path.join(img_dir, img_info["file_name"])
    if not os.path.exists(local_path):
        urllib.request.urlretrieve(img_info["coco_url"], local_path)
    print(f"Downloaded: {img_info['file_name']} (ID: {img_info['id']}, {img_info['width']}x{img_info['height']})")

Downloaded: 000000210273.jpg (ID: 210273, 640x428)
Downloaded: 000000254814.jpg (ID: 254814, 640x480)
Downloaded: 000000309391.jpg (ID: 309391, 640x384)
Downloaded: 000000490936.jpg (ID: 490936, 480x640)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, img_info in zip(axes, sample_images):
    img_id = img_info["id"]
    img_path = os.path.join(img_dir, img_info["file_name"])

    # Query annotations directly for this image_id
    matching_anns = [ann for ann in annotations if ann["image_id"] == img_id]

    image = Image.open(img_path).convert("RGB")
    ax.imshow(image)
    ax.set_title(f"Image ID: {img_id} | {img_info['file_name']} | Objects: {len(matching_anns)}", fontsize=10)
    ax.axis("off")

    for ann in matching_anns:
        x, y, w, h = ann["bbox"]
        label = cat_id_to_name.get(ann["category_id"], f"ID:{ann['category_id']}")

        rect = patches.Rectangle(
            (x, y), w, h,
            linewidth=2,
            edgecolor="#00FF00",
            facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(
            x, max(y - 4, 0),
            label,
            color="black",
            fontsize=8,
            fontweight="bold",
            bbox=dict(facecolor="#00FF00", alpha=0.85, pad=1.2, edgecolor="none")
        )

plt.tight_layout()
plt.show()

In [ ]:
verification_rows = []

for img_info in sample_images:
    img_id = img_info["id"]
    matching_anns = [ann for ann in annotations if ann["image_id"] == img_id]
    for ann in matching_anns[:3]:  # Inspect first 3 objects per sample image
        x, y, w, h = ann["bbox"]
        verification_rows.append({
            "image_id": img_id,
            "file_name": img_info["file_name"],
            "ann_id": ann["id"],
            "category_id": ann["category_id"],
            "category_name": cat_id_to_name[ann["category_id"]],
            "bbox_xywh": [round(v, 1) for v in ann["bbox"]],
            "computed_area": round(w * h, 1),
            "recorded_seg_area": round(ann["area"], 1),
            "within_bounds": (0 <= x) and (0 <= y) and (x + w <= img_info["width"]) and (y + h <= img_info["height"])
        })

verify_df = pd.DataFrame(verification_rows)
display(verify_df)

,image_id,file_name,ann_id,category_id,category_name,bbox_xywh,computed_area,recorded_seg_area,within_bounds
0,210273,000000210273.jpg,127213,2,bicycle,"[467.7, 178.5, 34.8, 17.3]",600.8,192.5,True
1,210273,000000210273.jpg,164381,6,bus,"[96.6, 189.1, 126.6, 129.5]",16385.8,14253.5,True
2,210273,000000210273.jpg,166669,6,bus,"[228.9, 133.9, 65.4, 88.5]",5787.2,5047.4,True
3,254814,000000254814.jpg,241548,2,bicycle,"[69.9, 269.2, 49.4, 27.1]",1337.1,728.4,True
4,254814,000000254814.jpg,241697,2,bicycle,"[496.8, 129.3, 32.8, 25.7]",843.5,337.5,True
5,254814,000000254814.jpg,241832,2,bicycle,"[265.3, 231.1, 19.7, 19.7]",388.1,259.8,True
6,309391,000000309391.jpg,124978,2,bicycle,"[423.9, 198.7, 28.8, 20.4]",586.5,334.6,True
7,309391,000000309391.jpg,139897,3,car,"[437.2, 120.6, 22.7, 19.1]",434.4,251.2,True
8,309391,000000309391.jpg,184647,1,person,"[79.6, 252.9, 24.1, 72.0]",1736.6,1040.2,True
9,490936,000000490936.jpg,133825,3,car,"[352.5, 106.8, 22.3, 36.7]",818.0,553.2,True


In [ ]:
crowd_counts = ann_df["iscrowd"].value_counts().rename(index={0: "Individual Instance (0)", 1: "Crowd Region (1)"})
print("=== Annotation Segmentation Mode Breakdown ===")
print(crowd_counts)

=== Annotation Segmentation Mode Breakdown ===
iscrowd
Individual Instance (0)    36335
Crowd Region (1)             446
Name: count, dtype: int64


In [ ]:
missing_indian_classes = [
    "auto-rickshaw / tuk-tuk",
    "e-rickshaw",
    "cycle-rickshaw",
    "tempo / light commercial three-wheeler",
    "stray cattle / road animals (Indian road context)",
    "Indian traffic signage & lane markings"
]

print("=== Domain Scope & Limitation Check ===")
print("1. Dataset Audited     : MS COCO 2017 Validation Set (General Western-centric photography).")
print("2. Covered Road Classes:", list(traffic_related["name"]))
print("3. Missing Road Classes:", missing_indian_classes)
print("4. Audit Conclusion    : COCO validates schema and baseline object detection pipelines,")
print("                         but its class distribution does NOT represent Indian traffic conditions.")

=== Domain Scope & Limitation Check ===
1. Dataset Audited     : MS COCO 2017 Validation Set (General Western-centric photography).
2. Covered Road Classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'traffic light', 'stop sign']
3. Missing Road Classes: ['auto-rickshaw / tuk-tuk', 'e-rickshaw', 'cycle-rickshaw', 'tempo / light commercial three-wheeler', 'stray cattle / road animals (Indian road context)', 'Indian traffic signage & lane markings']
4. Audit Conclusion    : COCO validates schema and baseline object detection pipelines,
                         but its class distribution does NOT represent Indian traffic conditions.


In [ ]:
summary_report = f"""
====================================================================
DLAC HANDOUT - WEEK 5 DATASET AUDIT SUMMARY (MS COCO val2017)
====================================================================
1. Schema Verification   : PASSED (Top-level keys: images, annotations, categories)
2. Total Images Audited  : {len(images):,}
3. Total Annotations     : {len(annotations):,}
4. Total Categories      : {len(categories)} (IDs span 1 to {cat_df['id'].max()}, non-contiguous)
5. BBox Format Verified  : [x_min, y_min, width, height] in pixel coordinates
6. Geometry Integrity    : {len(annotations) - len(bbox_errors):,} / {len(annotations):,} valid boxes ({len(bbox_errors)} anomalies)
7. Visual Verification   : Completed on {len(sample_images)} multi-object sample images
8. Domain Caveat         : Standard COCO benchmark only; requires domain-specific Indian
                           traffic datasets (e.g., IDD) before claiming Indian road representation.
====================================================================
"""
print(summary_report)


DLAC HANDOUT - WEEK 5 DATASET AUDIT SUMMARY (MS COCO val2017)
1. Schema Verification   : PASSED (Top-level keys: images, annotations, categories)
2. Total Images Audited  : 5,000
3. Total Annotations     : 36,781
4. Total Categories      : 80 (IDs span 1 to 90, non-contiguous)
5. BBox Format Verified  : [x_min, y_min, width, height] in pixel coordinates
6. Geometry Integrity    : 36,781 / 36,781 valid boxes (0 anomalies)
7. Visual Verification   : Completed on 4 multi-object sample images
8. Domain Caveat         : Standard COCO benchmark only; requires domain-specific Indian
                           traffic datasets (e.g., IDD) before claiming Indian road representation.

